# RSNA Knee — OFFLINE submission v2 (gold-honest CV + two-stage fine-tune)

Changes from the v1 EfficientNet-B0 notebook, in priority order of expected impact:

1. **Gold-stratified validation.** v1 K-folded *everything* (gold+weak together), so
   each fold's "gold_auc" was computed on whichever 7–15 gold studies happened to land
   in that fold — noisy enough to make epoch-to-epoch comparisons close to meaningless
   (see v1's own log: 0.77 → 0.65 → 0.75 → 0.74 → 0.70 within one fold). Here the 5 folds
   are built **only over the 58 gold studies**; all weak data is used in every fold's
   training (no longer wasted on folds where it lands in "val"). We also collect
   **out-of-fold (OOF) predictions across all 58 gold studies** and report one honest
   macro-AUC number for the whole pipeline, not five noisy per-fold numbers.
2. **Better pseudo-labels.** The v1 NLI aggregation (max over sentences *and* over
   hypothesis paraphrases) is prone to one spurious sentence/paraphrase match inflating
   a score. We now compare max-vs-mean-over-hypotheses **per label**, scored against
   gold, and keep whichever wins — plus an optional keyword-match signal blended in at
   a weight chosen by cross-validated grid search on gold (not just eyeballed).
3. **Two-stage training per fold.** Stage A trains on weak+gold (broad signal, same as
   v1). Stage B then **freezes the CNN backbone** and fine-tunes only the
   attention+classifier head on the gold-train split alone (clean labels, tiny data —
   freezing the backbone is what makes this not immediately overfit). Best epoch chosen
   by gold-val AUC within each fold.
4. **TTA at inference** (horizontal flip, averaged) on top of the existing 5-fold
   ensemble average.

**Expectation-setting, explicitly for the viva:** with only 58 gold-labeled studies,
treat any single AUC number — local or leaderboard — as a noisy estimate, not a precise
score. These changes aim to raise the *honest* macro AUC meaningfully and make the
validation itself trustworthy; they are not expected to reach 0.90.


In [1]:
import os, re, gc, time, glob, random, warnings
warnings.filterwarnings("ignore")
os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np, pandas as pd, pydicom, cv2, torch, torch.nn as nn, timm
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK_DIR = "/kaggle/working"
LABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
              "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
VIEWS = ["Sagittal","Coronal","Axial"]

# ---------------- OPTIMIZED-RUN CONFIG ----------------
N_WEAK    = 2500    # unlabeled (report-only) studies to auto-label + train on (of 4349 total)
EPOCHS    = 6
N_SLICES  = 16
IMG_SIZE  = 176
CROP      = 160
BATCH     = 8
LR        = 3e-4
SEED      = 42
N_FOLDS       = 5     # folds over the 58 GOLD studies only (weak data is in every fold's train)
STAGE_B_EPOCHS = 8    # frozen-backbone fine-tune epochs on gold-train only
STAGE_B_LR     = 5e-5 # smaller LR than stage A - we're only training the head now
TTA_FLIP       = True # horizontal-flip test-time augmentation at inference
NUM_WORKERS = min(4, os.cpu_count() or 1)
# -------------------------------------------------------

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
print("Device:", DEVICE, "| CPUs:", os.cpu_count())


Device: cuda | CPUs: 4


## 1. Locate inputs (no internet, no hard-coded dataset names)

In [2]:
import json, zipfile
SKIP = {"train_series", "test_series"}   # never walk the huge DICOM folders

def walk_input():
    for cur, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in SKIP]
        yield cur, dirs, files

DATA_DIR = next((cur for cur, _, files in walk_input()
                 if "train.csv" in files and "train_series.csv" in files), None)
assert DATA_DIR, "Competition data not found under /kaggle/input"

# NLI model: the folder that actually contains config.json + weights (datasets are nested)
NLI_DIR = next((cur for cur, _, files in walk_input()
                if "config.json" in files and ("model.safetensors" in files or "pytorch_model.bin" in files)
                and "competitions" not in cur), None)
assert NLI_DIR, "NLI model folder (config.json + weights) not found - is the deberta dataset attached?"

# must be a real NLI-finetuned model, not plain deberta-v3-large (random head -> garbage weak labels)
_labs = [str(v).lower() for v in json.load(open(f"{NLI_DIR}/config.json")).get("id2label", {}).values()]
assert any("entail" in l for l in _labs), (
    f"{NLI_DIR} has labels {_labs} - not an NLI model. Upload an NLI-finetuned checkpoint "
    "(e.g. MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli) as the dataset instead.")

# EfficientNet weights: Kaggle auto-extracts uploaded .pth zips into a folder -> re-zip so torch.load works
def find_effnet():
    for cur, _, files in walk_input():
        for f in files:
            if f.startswith("efficientnet") and f.endswith(".pth"): return os.path.join(cur, f)
        if "data.pkl" in files and "efficientnet" in cur: return cur
    return None

def as_pth(path):
    if path is None or os.path.isfile(path): return path
    out = f"{WORK_DIR}/efficientnet_b0_timm.pth"
    root = os.path.basename(path.rstrip("/"))
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for cur, _, files in os.walk(path):
            for f in files:
                full = os.path.join(cur, f)
                z.write(full, os.path.join(root, os.path.relpath(full, path)))
    return out

EFFNET_W = as_pth(find_effnet())

print("DATA_DIR  :", DATA_DIR); print("NLI_DIR   :", NLI_DIR); print("EFFNET_W  :", EFFNET_W)
if EFFNET_W is None:
    print("WARNING: efficientnet_b0 weights not found -> random init (much worse accuracy)")


DATA_DIR  : /kaggle/input/competitions/rsna-knee-abnormality-detection
NLI_DIR   : /kaggle/input/datasets/ad8thya/deberta-nil-offline/deberta_nil_offline
EFFNET_W  : /kaggle/working/efficientnet_b0_timm.pth


## 2. Load CSVs

In [3]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
train_series = pd.read_csv(f"{DATA_DIR}/train_series.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
test_series = pd.read_csv(f"{DATA_DIR}/test_series.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
print(train.shape, train_series.shape, test.shape, test_series.shape)

has_labels = train[LABEL_COLS].notna().all(axis=1)
gold_df = train[has_labels]
unl_df = train[~has_labels]
print(f"gold={len(gold_df)}  unlabeled={len(unl_df)}")

# subsample: ALL gold + N_WEAK random unlabeled studies
sub = pd.concat([gold_df, unl_df.sample(n=min(N_WEAK, len(unl_df)), random_state=SEED)]).reset_index(drop=True)
gold_mask = sub[LABEL_COLS].notna().all(axis=1).values
print("subsample:", len(sub), "| gold in it:", gold_mask.sum())

(4407, 14) (24371, 5) (3, 1) (15, 5)
gold=58  unlabeled=4349
subsample: 2558 | gold in it: 58


## 3. Weak labelling with the LOCAL NLI model
Same NLI scoring as v1 (report -> sentences -> NLI(sentence, hypothesis)), unchanged
below, since re-running it is the expensive part (~9 min) and the scoring logic itself
is fine — what we improve is how the per-sentence/per-hypothesis scores get
**aggregated into one score per label**, next cell.

In [4]:
def clean_report(t):
    if not isinstance(t, str): return ""
    t = re.sub(r"\[.*?\]", "", t)
    return re.sub(r"\s+", " ", t).strip()

def split_sentences(text, min_chars=3):
    parts = re.split(r"(?<=[.;!?])\s+|\n+", str(text))
    out, seen = [], set()
    for p in parts:
        p = p.strip(" .-"); k = p.lower()
        if len(p) >= min_chars and k not in seen:
            seen.add(k); out.append(p)
    return out if out else [str(text)[:1000] or "empty report"]

# expanded: 2-3 paraphrases per finding instead of 1, to catch more varied
# radiologist phrasing (still English; relies on mDeBERTa's multilingual
# cross-lingual NLI transfer for non-English reports, same as before)
HYPOTHESES = {
 "ACL": ["The anterior cruciate ligament is torn.",
         "There is an injury or rupture of the anterior cruciate ligament.",
         "The ACL shows a partial or complete tear."],
 "MCL": ["The medial collateral ligament is torn or sprained.",
         "There is an injury of the medial collateral ligament."],
 "Medial Meniscus": ["There is a tear of the medial meniscus.",
                      "The medial meniscus shows a tear or degenerative injury."],
 "Lateral Meniscus": ["There is a tear of the lateral meniscus.",
                       "The lateral meniscus shows a tear or degenerative injury."],
 "Medial OA": ["There is osteoarthritis of the medial femorotibial compartment.",
               "There is cartilage loss or degenerative change in the medial compartment."],
 "Lateral OA": ["There is osteoarthritis of the lateral femorotibial compartment.",
                "There is cartilage loss or degenerative change in the lateral compartment."],
 "PF OA": ["There is patellofemoral osteoarthritis or chondromalacia of the patella.",
           "There is cartilage loss in the patellofemoral joint."],
 "Effusion": ["There is a joint effusion.",
              "There is excess fluid in the knee joint."],
 "Synovitis": ["There is synovitis.",
               "There is inflammation of the synovial lining."],
 "Baker's": ["There is a Baker's cyst.",
             "There is a popliteal cyst."],
 "Contusion": ["There is a bone contusion or bone bruise.",
               "There is a muscle contusion."],
 "Fracture": ["There is a fracture.",
              "There is a break in the bone."],
}
HYP_FLAT = [(lab, h) for lab in LABEL_COLS for h in HYPOTHESES[lab]]

sub["sentences"] = sub["Report"].apply(clean_report).apply(split_sentences)
print("NLI pairs:", int(sub["sentences"].str.len().sum()) * len(HYP_FLAT))

@torch.no_grad()
def nli_scores(sentences, hyps, sent_batch=32, max_len=160):
    tok = AutoTokenizer.from_pretrained(NLI_DIR, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(NLI_DIR, local_files_only=True).to(DEVICE).eval()
    if DEVICE == "cuda": model.half()

    print("NLI labels:", model.config.id2label)
    l2i = {k.lower(): v for k, v in model.config.label2id.items()}
    if "entailment" in l2i and "contradiction" in l2i:
        cols = [l2i["entailment"], l2i["contradiction"]]
    elif "entailment" in l2i:
        cols = [l2i["entailment"]] + [i for i in range(model.config.num_labels) if i != l2i["entailment"]]
    else:
        cols = [0, 2]
    print("Using logit columns [entail, other]:", cols)

    H = len(hyps); out = np.zeros((len(sentences), H), np.float32)
    order = np.argsort([len(s) for s in sentences])
    for st in tqdm(range(0, len(order), sent_batch), desc="NLI"):
        idx = order[st:st + sent_batch]
        prem = [sentences[i] for i in idx for _ in range(H)]
        hyp = list(hyps) * len(idx)
        enc = tok(prem, hyp, truncation="only_first", max_length=max_len, padding=True, return_tensors="pt").to(DEVICE)
        lg = model(**enc).logits.float()
        out[idx] = torch.softmax(lg[:, cols], -1)[:, 0].view(len(idx), H).cpu().numpy()
    del model; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return out

flat, owner = [], []
for si, ss in enumerate(sub["sentences"]):
    flat.extend(ss); owner.extend([si] * len(ss))
owner = np.array(owner)

t0 = time.time()
sent_sc = nli_scores(flat, [h for _, h in HYP_FLAT])
print(f"NLI done in {(time.time()-t0)/60:.1f} min")

hyp_sc = np.zeros((len(sub), len(HYP_FLAT)), np.float32)
np.maximum.at(hyp_sc, owner, sent_sc)
study_scores = np.stack([hyp_sc[:, [j for j, (l, _) in enumerate(HYP_FLAT) if l == lab]].max(1) for lab in LABEL_COLS], 1)


NLI pairs: 952675


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

NLI labels: {0: 'entailment', 1: 'not_entailment'}
Using logit columns [entail, other]: [0, 1]


NLI:   0%|          | 0/1191 [00:00<?, ?it/s]

NLI done in 10.1 min


In [5]:

# --- NEW: pick aggregation + optional keyword blend using gold as ground truth ---
from sklearn.model_selection import KFold as _KFold

def safe_auc(y, s):
    return roc_auc_score(y, s) if len(np.unique(y)) == 2 else np.nan

Yg_check = sub.loc[gold_mask, LABEL_COLS].values.astype(int)
gold_pos = np.where(gold_mask.values if hasattr(gold_mask, "values") else gold_mask)[0]

# max-over-hypotheses (= v1's approach) vs mean-over-hypotheses, per label
score_max  = study_scores  # already computed above as max-over-hypotheses per label
score_mean = np.stack(
    [hyp_sc[:, [j for j, (l, _) in enumerate(HYP_FLAT) if l == lab]].mean(1) for lab in LABEL_COLS], 1
)

auc_max  = [safe_auc(Yg_check[:, j], score_max[gold_pos, j])  for j in range(len(LABEL_COLS))]
auc_mean = [safe_auc(Yg_check[:, j], score_mean[gold_pos, j]) for j in range(len(LABEL_COLS))]
use_mean = np.array([(m if not np.isnan(m) else -1) > (x if not np.isnan(x) else -1)
                      for m, x in zip(auc_mean, auc_max)])

print("Aggregation choice per label (gold AUC max vs mean):")
for j, lab in enumerate(LABEL_COLS):
    print(f"  {lab:18s} max={auc_max[j]:.3f}  mean={auc_mean[j]:.3f}  -> using {'mean' if use_mean[j] else 'max'}")

base_score = np.where(use_mean[None, :], score_mean, score_max)

# --- simple English keyword signal (helps mainly the ~40% English reports; harmless
#     elsewhere since it just won't fire on non-English text) ---
KEYWORDS = {
    "ACL": [r"\bacl\b.*tear", r"anterior cruciate.*(tear|rupture|injur)"],
    "MCL": [r"\bmcl\b", r"medial collateral ligament"],
    "Medial Meniscus": [r"medial meniscus.*(tear|injur)"],
    "Lateral Meniscus": [r"lateral meniscus.*(tear|injur)"],
    "Medial OA": [r"medial.*(osteoarthritis|compartment narrowing)"],
    "Lateral OA": [r"lateral.*(osteoarthritis|compartment narrowing)"],
    "PF OA": [r"patellofemoral.*(osteoarthritis|chondromalacia)"],
    "Effusion": [r"\beffusion\b", r"joint fluid"],
    "Synovitis": [r"synoviti?s"],
    "Baker's": [r"baker'?s? cyst", r"popliteal cyst"],
    "Contusion": [r"contusion", r"bone bruis"],
    "Fracture": [r"fracture"],
}
KW_RE = {lab: re.compile("|".join(pats), re.IGNORECASE) for lab, pats in KEYWORDS.items()}

def keyword_score(report):
    text = clean_report(report)
    return np.array([1.0 if KW_RE[lab].search(text) else 0.0 for lab in LABEL_COLS], np.float32)

kw_scores = np.stack([keyword_score(r) for r in sub["Report"]], 0)

# pick a blend weight alpha (final = (1-alpha)*base_score + alpha*kw_scores) via
# 5-fold CV *on gold only* so we don't just overfit alpha to all 58 points at once
alphas = np.linspace(0, 0.5, 6)
kf_a = _KFold(n_splits=5, shuffle=True, random_state=SEED)
alpha_scores = []
for a in alphas:
    blended_gold = (1 - a) * base_score[gold_pos] + a * kw_scores[gold_pos]
    fold_aucs = []
    for tr_i, va_i in kf_a.split(gold_pos):
        va = gold_pos[va_i]
        aucs = [safe_auc(Yg_check[va_i][:, j], blended_gold[va_i, j]) for j in range(len(LABEL_COLS))]
        fold_aucs.append(np.nanmean(aucs))
    alpha_scores.append(np.mean(fold_aucs))

best_alpha = alphas[int(np.argmax(alpha_scores))]
print("\nAlpha (keyword blend weight) CV macro AUC:", dict(zip(alphas.round(2), np.round(alpha_scores, 4))))
print("Selected alpha:", best_alpha)

study_scores = (1 - best_alpha) * base_score + best_alpha * kw_scores
final_auc = [safe_auc(Yg_check[:, j], study_scores[gold_pos, j]) for j in range(len(LABEL_COLS))]
print("\nFinal labeller macro AUC on gold (post aggregation+blend choice):", round(float(np.nanmean(final_auc)), 4))
print("(compare to v1's raw max-only macro AUC — check cell above for that number)")


Aggregation choice per label (gold AUC max vs mean):
  ACL                max=0.640  mean=0.636  -> using max
  MCL                max=0.660  mean=0.567  -> using max
  Medial Meniscus    max=0.781  mean=0.769  -> using max
  Lateral Meniscus   max=0.735  mean=0.686  -> using max
  Medial OA          max=0.633  mean=0.616  -> using max
  Lateral OA         max=0.739  mean=0.798  -> using mean
  PF OA              max=0.607  mean=0.593  -> using max
  Effusion           max=0.570  mean=0.565  -> using max
  Synovitis          max=0.573  mean=0.589  -> using mean
  Baker's            max=0.735  mean=0.765  -> using mean
  Contusion          max=0.655  mean=0.644  -> using max
  Fracture           max=0.593  mean=0.611  -> using mean

Alpha (keyword blend weight) CV macro AUC: {np.float64(0.0): np.float64(0.6695), np.float64(0.1): np.float64(0.6587), np.float64(0.2): np.float64(0.6479), np.float64(0.3): np.float64(0.6416), np.float64(0.4): np.float64(0.6381), np.float64(0.5): np.float64(0

In [6]:
def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4); return np.log(p / (1 - p))

# Platt calibration per label (falls back to raw score if gold has <2 positives/negatives)
soft = np.zeros_like(study_scores)
for j, lab in enumerate(LABEL_COLS):
    y = Yg_check[:, j]
    if y.sum() >= 2 and (1 - y).sum() >= 2:
        cal = LogisticRegression().fit(logit(study_scores[gold_pos][:, [j]]), y)
        soft[:, j] = cal.predict_proba(logit(study_scores[:, [j]]))[:, 1]
    else:
        soft[:, j] = study_scores[:, j]
soft[gold_pos] = Yg_check           # gold studies keep true labels

labels_df = pd.DataFrame(soft, columns=LABEL_COLS)
labels_df.insert(0, "StudyInstanceUID", sub["StudyInstanceUID"].values)
labels_df["is_gold"] = False
labels_df.loc[gold_pos, "is_gold"] = True
print(pd.DataFrame({"gold": Yg_check.mean(0), "weak": soft[~labels_df["is_gold"]].mean(0)}, index=LABEL_COLS).round(3))


                   gold   weak
ACL               0.414  0.362
MCL               0.155  0.123
Medial Meniscus   0.448  0.378
Lateral Meniscus  0.397  0.297
Medial OA         0.259  0.241
Lateral OA        0.190  0.155
PF OA             0.362  0.321
Effusion          0.603  0.580
Synovitis         0.466  0.426
Baker's           0.207  0.188
Contusion         0.328  0.274
Fracture          0.310  0.248


## 4. MRI loading (decoded once into RAM for the subsample — no disk cache)

In [7]:
def pick_view_series(rows, plane):
    r = rows[rows["Anatomical_Plane"] == plane]
    if len(r) == 0: return None
    f = r[r["Fluid_Sensitive"] == 1]
    return (f if len(f) else r).iloc[0]["SeriesInstanceUID"]

def sorted_files(d):
    files = [f for f in os.listdir(d) if not f.startswith(".")]
    stems = [os.path.splitext(f)[0] for f in files]
    if all(s.isdigit() for s in stems):
        return [f for _, f in sorted(zip(map(int, stems), files))]
    keyed = []
    for f in files:
        try: n = int(pydicom.dcmread(os.path.join(d, f), stop_before_pixels=True).InstanceNumber)
        except Exception: n = 0
        keyed.append((n, f))
    return [f for _, f in sorted(keyed)]

def load_volume_u8(split, study, series):
    d = f"{DATA_DIR}/{split}_series/{study}/{series}"
    files = sorted_files(d)
    idx = np.linspace(0, len(files) - 1, N_SLICES).round().astype(int)
    imgs = []
    for i in idx:
        try:
            a = pydicom.dcmread(os.path.join(d, files[i])).pixel_array.astype(np.float32)
            if a.ndim == 3: a = a[a.shape[0] // 2]
            a = cv2.resize(a, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        except Exception:
            a = np.zeros((IMG_SIZE, IMG_SIZE), np.float32)
        imgs.append(a)
    v = np.stack(imgs)
    lo, hi = np.percentile(v, [1, 99.5])
    return (np.clip((v - lo) / (hi - lo + 1e-6), 0, 1) * 255).astype(np.uint8)

def build_study(split, study, rows):
    vol = np.zeros((len(VIEWS), N_SLICES, IMG_SIZE, IMG_SIZE), np.uint8)
    mask = np.zeros(len(VIEWS), np.float32)
    for v, plane in enumerate(VIEWS):
        s = pick_view_series(rows, plane)
        if s is None: continue
        try: vol[v] = load_volume_u8(split, study, s); mask[v] = 1.0
        except Exception: pass
    if mask.sum() == 0: mask[0] = 1.0
    return vol, mask

ser_train = {k: g for k, g in train_series.groupby("StudyInstanceUID")}
ser_test = {k: g for k, g in test_series.groupby("StudyInstanceUID")}
EMPTY = train_series.iloc[0:0]

t0 = time.time()
res = Parallel(n_jobs=os.cpu_count(), backend="loky")(
    delayed(build_study)("train", u, ser_train.get(u, EMPTY)) for u in tqdm(sub["StudyInstanceUID"], desc="decode train"))
VOLS = np.stack([r[0] for r in res]); MASKS = np.stack([r[1] for r in res]); del res
print(f"decoded {VOLS.shape} in {(time.time()-t0)/60:.1f} min | RAM {VOLS.nbytes/1e9:.2f} GB")
print("all-zero volumes:", int((VOLS.reshape(len(VOLS), -1).max(1) == 0).sum()), "(should be ~0)")

decode train:   0%|          | 0/2558 [00:00<?, ?it/s]

decoded (2558, 3, 16, 176, 176) in 23.1 min | RAM 3.80 GB
all-zero volumes: 0 (should be ~0)


## 5. Model: 2.5D EfficientNet-B0 + per-label attention over slices/views (offline weights)
Unchanged from v1, plus two small additions: `freeze_encoder()` / `unfreeze_encoder()`
for Stage B, and `predict` now supports horizontal-flip TTA.

In [8]:
def crop(vol, c, rand):
    H, W = vol.shape[-2:]
    if rand: y0, x0 = np.random.randint(0, H - c + 1), np.random.randint(0, W - c + 1)
    else:    y0, x0 = (H - c) // 2, (W - c) // 2
    return vol[..., y0:y0 + c, x0:x0 + c]

class TrainDS(torch.utils.data.Dataset):
    def __init__(self, idx, targets, train): self.idx, self.t, self.train = list(idx), targets.astype(np.float32), train
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        v = torch.from_numpy(VOLS[self.idx[i]]).float() / 255.0
        v = crop(v, CROP, self.train)
        if self.train: v = (v * np.random.uniform(0.9, 1.1) + np.random.uniform(-0.05, 0.05)).clamp(0, 1)
        return v, torch.from_numpy(MASKS[self.idx[i]]), torch.from_numpy(self.t[i])

class TestDS(torch.utils.data.Dataset):      # decodes DICOMs on the fly inside DataLoader workers
    def __init__(self, uids): self.uids = list(uids)
    def __len__(self): return len(self.uids)
    def __getitem__(self, i):
        u = self.uids[i]
        vol, mask = build_study("test", u, ser_test.get(u, EMPTY))
        return crop(torch.from_numpy(vol).float() / 255.0, CROP, False), torch.from_numpy(mask)

class MultiViewMIL(nn.Module):
    def __init__(self, weights=None, n_views=3, n_labels=len(LABEL_COLS), drop=0.3):
        super().__init__()
        # EfficientNet-B0: 5.3M params, higher ImageNet accuracy than ResNet18 (11.7M params)
        self.encoder = timm.create_model("efficientnet_b0", pretrained=False, num_classes=0, in_chans=3)
        if weights:
            sd = torch.load(weights, map_location="cpu")
            sd = sd.get("state_dict", sd)
            print("load efficientnet_b0:", self.encoder.load_state_dict(sd, strict=False))
        D = self.encoder.num_features
        self.view_emb = nn.Parameter(torch.zeros(n_views, D))
        self.att = nn.Sequential(nn.Linear(D, 256), nn.Tanh(), nn.Linear(256, n_labels))
        self.drop = nn.Dropout(drop)
        self.cls_w = nn.Parameter(torch.randn(n_labels, D) * 0.01)
        self.cls_b = nn.Parameter(torch.zeros(n_labels))

    def freeze_encoder(self):
        for p in self.encoder.parameters(): p.requires_grad_(False)

    def unfreeze_encoder(self):
        for p in self.encoder.parameters(): p.requires_grad_(True)

    @staticmethod
    def to_25d(x):
        prev = torch.cat([x[:, :, :1], x[:, :, :-1]], 2); nxt = torch.cat([x[:, :, 1:], x[:, :, -1:]], 2)
        return torch.stack([prev, x, nxt], 3)

    def forward(self, x, view_mask):
        B, V, S, H, W = x.shape
        x = (self.to_25d(x) - 0.45) / 0.225
        f = self.encoder(x.reshape(B * V * S, 3, H, W)).reshape(B, V, S, -1) + self.view_emb[None, :, None, :]
        tok = f.reshape(B, V * S, -1)
        tm = view_mask[:, :, None].expand(B, V, S).reshape(B, V * S)
        a = self.att(tok).float().masked_fill(tm[..., None] == 0, -1e4)
        a = torch.softmax(a, 1)
        pooled = torch.einsum("btl,btd->bld", a, self.drop(tok).float())
        return (pooled * self.cls_w).sum(-1) + self.cls_b

def safe_auc(y, s):
    return roc_auc_score(y, s) if len(np.unique(y)) == 2 else np.nan

def macro_auc(y, p):
    a = [safe_auc(y[:, j], p[:, j]) for j in range(y.shape[1])]
    return np.nanmean(a), a

@torch.no_grad()
def predict(model, loader, tta_flip=False):
    model.eval(); out = []
    for b in loader:
        x, m = b[0].to(DEVICE), b[1].to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            lg = model(x, m)
            if tta_flip:
                lg = lg + model(torch.flip(x, dims=[-1]), m)
                lg = lg / 2
        out.append(torch.sigmoid(lg.float()).cpu().numpy())
    return np.concatenate(out)


## 6. Train — gold-stratified 5-fold, two-stage per fold

**Stage A** (same as v1): train on weak-full + gold-train for `EPOCHS` epochs.
**Stage B** (new): freeze the CNN backbone, fine-tune only the attention+classifier
head on gold-train alone for `STAGE_B_EPOCHS` epochs at a lower LR — clean labels,
tiny data, so freezing the backbone is what keeps this from immediately overfitting.
Every epoch across both stages is checked against gold-val AUC; the single best
checkpoint per fold is kept. OOF gold predictions are collected across all 5 folds to
give one honest macro-AUC estimate for the whole pipeline.

In [9]:

Y = labels_df[LABEL_COLS].values.astype(np.float32)
N = len(sub)
weak_idx = np.where(~labels_df["is_gold"].values)[0]
gold_idx_all = np.where(labels_df["is_gold"].values)[0]
print(f"weak studies (always in every fold's train): {len(weak_idx)} | gold studies (5-fold split): {len(gold_idx_all)}")

def make_loader(idx, train, batch=None):
    bs = batch or BATCH
    # drop_last only when training AND there'd be enough left for >=1 full batch,
    # so a small Stage-B gold-train split never ends up with zero batches
    dl = train and (len(idx) // bs >= 1)
    return torch.utils.data.DataLoader(TrainDS(idx, Y[idx], train), batch_size=bs, shuffle=train,
                                       num_workers=NUM_WORKERS, pin_memory=True, drop_last=dl)

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_ckpts, fold_scores = [], []
oof_pred = np.zeros((len(gold_idx_all), len(LABEL_COLS)), np.float32)
oof_true = Y[gold_idx_all].astype(int)
gold_pos_in_all = {g: i for i, g in enumerate(gold_idx_all)}

t_start = time.time()
for fold, (gtr, gva) in enumerate(kf.split(gold_idx_all)):
    gold_train = gold_idx_all[gtr]
    gold_val   = gold_idx_all[gva]
    print(f"\n===== FOLD {fold+1}/{N_FOLDS} ===== gold_train={len(gold_train)} gold_val={len(gold_val)}")

    stageA_idx = np.concatenate([weak_idx, gold_train])
    tr_ld_A   = make_loader(stageA_idx, True)
    gold_val_ld = make_loader(gold_val, False)

    model = MultiViewMIL(EFFNET_W).to(DEVICE)
    ckpt_path = f"{WORK_DIR}/best_model_fold{fold}.pth"
    best = -1

    # ---------------- Stage A: broad pretrain on weak + gold-train ----------------
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    total = EPOCHS * len(tr_ld_A)
    sched = get_cosine_schedule_with_warmup(opt, int(0.05 * total), total)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
    crit = nn.BCEWithLogitsLoss()

    for ep in range(EPOCHS):
        model.train(); run = 0
        for x, m, y in tqdm(tr_ld_A, desc=f"fold{fold} stageA epoch{ep+1}/{EPOCHS}"):
            x, m, y = x.to(DEVICE), m.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
                lg = model(x, m)
            loss = crit(lg.float(), y)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run += loss.item()

        val_auc, _ = macro_auc(Y[gold_val].astype(int), predict(model, gold_val_ld))
        elapsed = (time.time() - t_start) / 60
        print(f"fold{fold} stageA ep{ep+1} loss={run/len(tr_ld_A):.4f} gold_auc={val_auc:.4f} | {elapsed:.1f}m elapsed")
        if np.nan_to_num(val_auc) >= best:
            best = np.nan_to_num(val_auc)
            torch.save(model.state_dict(), ckpt_path)

    # ---------------- Stage B: frozen-backbone fine-tune on gold-train only ----------------
    model.freeze_encoder()
    tr_ld_B = make_loader(gold_train, True, batch=min(BATCH, max(2, len(gold_train))))
    opt_b = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                               lr=STAGE_B_LR, weight_decay=1e-2)
    total_b = STAGE_B_EPOCHS * max(1, len(tr_ld_B))
    sched_b = get_cosine_schedule_with_warmup(opt_b, int(0.1 * total_b), total_b)

    for ep in range(STAGE_B_EPOCHS):
        model.train(); run = 0
        for x, m, y in tr_ld_B:
            x, m, y = x.to(DEVICE), m.to(DEVICE), y.to(DEVICE)
            opt_b.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
                lg = model(x, m)
            loss = crit(lg.float(), y)
            scaler.scale(loss).backward(); scaler.unscale_(opt_b)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt_b); scaler.update(); sched_b.step(); run += loss.item()

        val_auc, _ = macro_auc(Y[gold_val].astype(int), predict(model, gold_val_ld))
        elapsed = (time.time() - t_start) / 60
        print(f"fold{fold} stageB ep{ep+1} loss={run/max(1,len(tr_ld_B)):.4f} gold_auc={val_auc:.4f} | {elapsed:.1f}m elapsed")
        if np.nan_to_num(val_auc) >= best:
            best = np.nan_to_num(val_auc)
            torch.save(model.state_dict(), ckpt_path)
    model.unfreeze_encoder()

    # reload this fold's best checkpoint (could be from stage A or B) and record OOF preds
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    val_pred = predict(model, gold_val_ld, tta_flip=TTA_FLIP)
    for local_i, g in enumerate(gold_val):
        oof_pred[gold_pos_in_all[g]] = val_pred[local_i]

    fold_ckpts.append(ckpt_path)
    fold_scores.append(best)
    del model; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

print(f"\nPer-fold best gold_auc: {[round(s, 4) for s in fold_scores]}")
print(f"Mean of per-fold bests: {np.mean(fold_scores):.4f}  <- same style of number v1 reported (still noisy, N~12/fold)")

oof_auc, oof_per_label = macro_auc(oof_true, oof_pred)
print(f"\n*** Honest OOF macro AUC across all 58 gold studies: {oof_auc:.4f} ***")
print(pd.Series(oof_per_label, index=LABEL_COLS).round(3))
print("^ this is the number to quote in your viva/report, not the per-fold mean above.")

del VOLS; gc.collect()


weak studies (always in every fold's train): 2500 | gold studies (5-fold split): 58

===== FOLD 1/5 ===== gold_train=46 gold_val=12
load efficientnet_b0: _IncompatibleKeys(missing_keys=[], unexpected_keys=['classifier.weight', 'classifier.bias'])


fold0 stageA epoch1/6:   0%|          | 0/318 [00:00<?, ?it/s]

fold0 stageA ep1 loss=0.5824 gold_auc=0.5717 | 3.3m elapsed


fold0 stageA epoch2/6:   0%|          | 0/318 [00:00<?, ?it/s]

fold0 stageA ep2 loss=0.5673 gold_auc=0.5548 | 6.4m elapsed


...

fold4 stageA ep5 loss=0.5519 gold_auc=0.6263 | 98.6m elapsed


fold4 stageA epoch6/6:   0%|          | 0/318 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c74e67416c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7c74e67416c0>  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
if w.is_alive():
      File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    self._shutdown_workers()assert self._parent_pid == os.getpid(), 'can only test a child process'

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
AssertionError    : can only test a child processif w.is_alive():

  File "/usr/lib/

fold4 stageA ep6 loss=0.5496 gold_auc=0.6378 | 101.7m elapsed
fold4 stageB ep1 loss=0.4363 gold_auc=0.6283 | 101.7m elapsed
fold4 stageB ep2 loss=0.4211 gold_auc=0.6318 | 101.8m elapsed
fold4 stageB ep3 loss=0.4251 gold_auc=0.6346 | 101.8m elapsed
fold4 stageB ep4 loss=0.4147 gold_auc=0.6313 | 101.8m elapsed
fold4 stageB ep5 loss=0.4260 gold_auc=0.6306 | 101.9m elapsed
fold4 stageB ep6 loss=0.4196 gold_auc=0.6311 | 101.9m elapsed
fold4 stageB ep7 loss=0.4162 gold_auc=0.6278 | 102.0m elapsed
fold4 stageB ep8 loss=0.4235 gold_auc=0.6306 | 102.0m elapsed

Per-fold best gold_auc: [np.float64(0.6137), np.float64(0.6834), np.float64(0.68), np.float64(0.7379), np.float64(0.665)]
Mean of per-fold bests: 0.6760  <- same style of number v1 reported (still noisy, N~12/fold)

*** Honest OOF macro AUC across all 58 gold studies: 0.6367 ***
ACL                 0.567
MCL                 0.608
Medial Meniscus     0.498
Lateral Meniscus    0.660
Medial OA           0.586
Lateral OA          0.747
PF OA

0

## 7. Inference on test -> TTA + ensembled across all fold models -> `/kaggle/working/submission.csv`

In [10]:
test_ld = torch.utils.data.DataLoader(TestDS(test["StudyInstanceUID"]), batch_size=BATCH, shuffle=False,
                                      num_workers=NUM_WORKERS, pin_memory=True)

t0 = time.time()
fold_preds = []
for ckpt_path in fold_ckpts:
    m = MultiViewMIL(None).to(DEVICE)   # weights=None: checkpoint already has fine-tuned backbone
    m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    fold_preds.append(predict(m, test_ld, tta_flip=TTA_FLIP))
    del m; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

pred = np.mean(fold_preds, axis=0)   # ensemble: average probabilities across folds
print(f"test inference ({len(fold_ckpts)}-fold ensemble) {(time.time()-t0)/60:.1f} min")

submission = pd.DataFrame(pred, columns=LABEL_COLS)
submission.insert(0, "StudyInstanceUID", test["StudyInstanceUID"].values)
submission = submission[sample_submission.columns]
submission[LABEL_COLS] = submission[LABEL_COLS].fillna(0.5).clip(0, 1)
assert len(submission) == len(test) and list(submission.columns) == list(sample_submission.columns)
submission.to_csv(f"{WORK_DIR}/submission.csv", index=False)
print(submission.shape); submission.head()


test inference (5-fold ensemble) 0.3 min
(3, 13)


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.374746,0.122737,0.341053,0.326311,0.264207,0.230768,0.324200,0.577257,0.429942,0.171772,0.326691,0.291650
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.399705,0.181894,0.602855,0.469476,0.394078,0.372025,0.433777,0.644511,0.528573,0.336963,0.367524,0.283929
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.366099,0.148663,0.488183,0.293838,0.303411,0.233935,0.392590,0.593036,0.427813,0.166511,0.264515,0.237270


## 8. What to try next if you have more runtime budget

Roughly in order of expected payoff per hour spent, given the 58-gold-label ceiling:

1. **Get more gold labels if at all possible.** This is the single highest-leverage
   thing in this competition — 58 is a very small clean set, and every other change
   here is working around that constraint rather than removing it.
2. **Full weak-data training** (raise `N_WEAK` toward the full 4,349, runtime
   permitting) — more pretraining signal for Stage A, even if noisy.
3. **CLIP as an *extra* ensemble member**, not a replacement: pre-download a CLIP
   ViT-B/32 checkpoint into a Kaggle Dataset (same pattern as `efficientnet_b0_timm.pth`),
   train a second, independent MultiViewMIL using it as the backbone, and average its
   predictions with this EfficientNet model's at inference — ensembling two differently-
   biased models usually helps more than swapping one backbone for another.
4. **More slices/higher resolution** (`N_SLICES`, `IMG_SIZE`) if your session's time
   budget allows — currently 16 slices at 176px per view is a fairly aggressive
   compute-saving subsample of each MRI series (which average ~34 slices).
5. Try an **ordinal/weighted BCE loss** reflecting how confident each weak label is
   (`soft` targets already carry some of this via calibration, but per-sample loss
   weighting by distance-from-gold could help further).
